# Phase 4 -- The MIMO Latency Loop
### *Nine Meters of Silence*, Chapter 1: "Thermal Suicide"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rjmachauthor/nine-meters-of-silence/blob/main/ch01/notebooks/phase4_mimo_latency.ipynb)

**The claim in the book:** the helicopter's roll isn't just reacting to the suspended cargo -- it's actively driving it. A software patch introduced a 0.2-second micro-latency into the flight computer's control loop. Because of the delay, the system executes a positive feedback loop: pilot-induced oscillation, with no pilot -- the code fighting its own airframe.

This notebook simulates the real mechanism: a suspended cargo mass coupled to helicopter roll dynamics, corrected by a delayed feedback controller. Watch what a 0.2-second delay -- something that sounds too small to matter -- actually does.

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/rjmachauthor/nine-meters-of-silence.git
    sys.path.insert(0, 'nine-meters-of-silence/ch01')
else:
    sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from physics.mimo_latency import simulate_roll_cargo_system, growth_rate

In [ ]:
# Run both scenarios: no delay (the system as designed) vs. the book's stated 0.2s delay
stable_run = simulate_roll_cargo_system(delay_s=0.0, duration_s=12.0)
unstable_run = simulate_roll_cargo_system(delay_s=0.2, duration_s=12.0)

print(f"Growth rate at 0.0s delay:  {growth_rate(stable_run['phi'], stable_run['time']):+.3f} /s  (settles)")
print(f"Growth rate at 0.2s delay:  {growth_rate(unstable_run['phi'], unstable_run['time']):+.3f} /s  (diverges)")

In [ ]:
# Side-by-side animation: same disturbance, same aircraft, only the control delay differs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, title in [(ax1, 'No delay (0.0s)'), (ax2, "Book's stated delay (0.2s)")]:
    ax.set_xlim(0, 12)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Roll angle (rad)')
    ax.set_title(title)

# Clip the unstable run's y-axis to something readable -- it truly does blow up
# to absurd values if left unscaled, which is itself part of the point.
ax1.set_ylim(-0.05, 0.05)
ax2.set_ylim(-2, 2)

line1, = ax1.plot([], [], color='seagreen', lw=1.5)
line2, = ax2.plot([], [], color='crimson', lw=1.5)

n_frames = 150
# only show the unstable run up to where it's still a readable oscillation,
# not the full numerical blowup
clip_idx = np.searchsorted(unstable_run['time'], 9.0)
frame_idx_1 = np.linspace(0, len(stable_run['time']) - 1, n_frames).astype(int)
frame_idx_2 = np.linspace(0, clip_idx, n_frames).astype(int)

def update(i):
    idx1 = frame_idx_1[i]
    idx2 = frame_idx_2[i]
    line1.set_data(stable_run['time'][:idx1+1], stable_run['phi'][:idx1+1])
    line2.set_data(unstable_run['time'][:idx2+1], unstable_run['phi'][:idx2+1])
    return line1, line2

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.tight_layout()
plt.close(fig)
HTML(ani.to_jshtml())

## Try it yourself
Same disturbance, same aircraft -- only the delay changes. Try values between the two above and watch where the growth rate actually crosses from settling to diverging.

In [ ]:
# --- Play with this ---
my_delay_seconds = 0.2   # book value: 0.2. Try 0.0, 0.05, 0.1, 0.15...
# ------------------------

r = simulate_roll_cargo_system(delay_s=my_delay_seconds, duration_s=15.0)
gr = growth_rate(r['phi'], r['time'])
verdict = "DIVERGES (unstable)" if gr > 0.05 else "settles (stable)"
print(f"At {my_delay_seconds}s delay: growth rate = {gr:+.3f}/s -> {verdict}")